# Phase 1: Understanding the data

Inspect the files, check their keys and joins, and prepare a provisional study subset. This notebook keeps all recorded sales days, including days without a matching price.

### Imports

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

print("numpy :", np.__version__)
print("pandas:", pd.__version__)

### Q1: What files do we actually have?

### Load the data

In [ ]:
DATA_DIR = Path("../data")
sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv", parse_dates=["date"])
prices = pd.read_csv(DATA_DIR / "sell_prices.csv")

for name, table in [("sales", sales), ("calendar", calendar), ("prices", prices)]:
    print(f"Table ({name}) has shape: {table.shape}")

In [ ]:
# See what are the data inside
# Since sales table has so many cols, I will just look at first 10 cols and last 10 cols for now

print("Sales Table:")
display(sales.iloc[:, :10].head())
display(sales.iloc[:, -10:].head())
display(sales.iloc[:, :10].tail())
print("Calendar Table:")
display(calendar.head())
print("Price Table:")
display(prices.head())

We have three tables:
- **Sales:** One row represents one item at one store, with daily unit sales in `d_1` through `d_1913`. The six identifier columns describe the series. For example, `HOBBIES_1_001_CA_1_validation` identifies an item-store series with a dataset suffix; it does not contain the daily sales values.
- **Calendar:** One row represents one calendar day. `d` connects the day labels in sales to actual dates. `wm_yr_wk` connects each day to a Walmart week. I will investigate the event and SNAP columns below.
- **Prices:** One row represents one item at one store in one Walmart week. `sell_price` is the weekly average selling price.

Our forecasting unit is **one item-store series**, and the target is its daily unit sales.

There are two related hierarchies: **State → Store** and **Category → Department → Item**. An item-store series combines the product and location. Now I want to see the unique values in these columns.

In [ ]:
sales_day_cols = [col for col in sales.columns if col.startswith("d_")]
print("Total sales days:", len(sales_day_cols))
sales.iloc[:, :10].info()
calendar.info()
prices.info()

In [ ]:
# Get categorical columns
sales_cat_cols = sales.select_dtypes(include=['object', 'string']).columns
display(sales[sales_cat_cols].nunique())

I will skip listing all `id` and `item_id` values because there are too many. I will inspect the other unique values.

In [ ]:
for col in sales_cat_cols:
    if col not in ('id', 'item_id'):
        print(f"{col} : {sales[col].unique()}")

In [ ]:
# See hierarchy between category and department
sales.groupby('cat_id')['dept_id'].unique()

In [ ]:
# See hierarchy between state and store
sales.groupby('state_id')['store_id'].unique()

In [ ]:
# Creates a clean matrix table of Category/Department vs State/Store
pd.crosstab([sales['cat_id'], sales['dept_id']], [sales['state_id'], sales['store_id']], margins=True)


Now I have a matrix showing the **number of item-store series** by department and store. These are counts of series, not total units sold. I can investigate total sales later; for now, I am checking the dataset structure.

For the calendar table, I checked the [M5 competitors’ guide](https://storage.googleapis.com/kaggle-forum-message-attachments/772349/15032/M5-Competitors-Guide-Final-10-March-2020.pdf):
- `wm_yr_wk` identifies a Walmart week. I will use the supplied mapping instead of trying to construct dates from the digits. Walmart week years need not match calendar years at year boundaries.
- `event_name_1` and `event_type_1` describe the first listed event. The `_2` columns describe a second event on the same date. Missing values mean no event is listed in that slot.
- SNAP means **Supplemental Nutrition Assistance Program**. Each state’s binary flag indicates whether SNAP purchases are allowed on that date; it is not a payout amount.
- In the displayed calendar, `wday=1` corresponds to Saturday. I should not assume Python’s weekday numbering is the same.

In [ ]:
# Find missing values

for name, table in [("sales", sales), ("calendar", calendar), ("prices", prices)]:
    missing_counts = table.isna().sum()
    # Only keep and print columns where missing values exist
    actual_missing = missing_counts[missing_counts > 0]
    print(f"Table ({name}):")
    if actual_missing.empty:
        print("No missing values found.")
    else:
        print(actual_missing)
    print()


Missing values in the event columns are expected when no event is listed. A raw table can have no empty fields and still lack records that we expect to find during a join.

In [ ]:
# Full-row duplicate checks for the smaller calendar table.
# Key checks below are more useful than comparing every sales-day column.
print("Calendar fully duplicated rows:", calendar.duplicated().sum())

Next, I noticed that checking fully duplicated rows is not enough. For joins, I need to check the specific key columns:
- Sales: `item_id` and `store_id` identify one series in the wide table.
- Calendar: `d` identifies one day; `date` should also be unique.
- Prices: `store_id`, `item_id`, and `wm_yr_wk` identify one price observation.

Several sales-day rows can match the same weekly price record.

In [ ]:
print("Sales duplicate item-store keys:", sales.duplicated(["item_id", "store_id"]).sum())
print("Calendar duplicate day keys:", calendar.duplicated(["d"]).sum())
print("Calendar duplicate dates:", calendar.duplicated(["date"]).sum())
print("Prices duplicate keys:", prices.duplicated(["store_id", "item_id", "wm_yr_wk"]).sum())

assert not sales[["item_id", "store_id"]].isna().any().any()
assert not sales.duplicated(["item_id", "store_id"]).any()
assert not calendar[["d", "date"]].isna().any().any()
assert calendar["d"].is_unique and calendar["date"].is_unique
assert not prices[["store_id", "item_id", "wm_yr_wk"]].isna().any().any()
assert not prices.duplicated(["store_id", "item_id", "wm_yr_wk"]).any()
print("Unique item-store combinations:", len(sales[["item_id", "store_id"]].drop_duplicates()))

### Explore the calendar and the available sales history

In [ ]:
print("Calendar date range:", calendar["date"].min().date(), "to", calendar["date"].max().date())
assert set(sales_day_cols).issubset(set(calendar["d"]))
sales_calendar = calendar.loc[calendar["d"].isin(sales_day_cols)].sort_values("date")
print("Observed sales date range:", sales_calendar["date"].min().date(), "to", sales_calendar["date"].max().date())
print("Observed sales days:", len(sales_calendar))
print("Approximate years of sales history:", round(len(sales_calendar) / 365.25, 2))
print("Calendar days beyond the sales columns:", len(calendar) - len(sales_calendar))

In [ ]:
display(calendar['event_name_1'].unique())
display(calendar['event_name_2'].unique())

In [ ]:
# Creates a clean matrix showing event type counts per year
pd.crosstab(calendar['event_type_1'], calendar['year'], margins=True)

In [ ]:
pd.crosstab(calendar['event_type_2'], calendar['year'], margins=True)

Now I understand more about the calendar table. The event crosstabs count recorded event slots, not sales. The first and last calendar years are partial, so their totals are not directly comparable with full years.

In [ ]:
print(f"Min Price: {prices['sell_price'].min()} \nMax Price: {prices['sell_price'].max()}")

## Provisional study scope

While it is fun to explore the dataset, for now I will narrow my working scope to **one store (`CA_1`) and two calendar years (2014 and 2015)** to make the next steps easier to manage.


In [ ]:
# Scope the Calendar down to the years 2014 and 2015
scoped_calendar = calendar[calendar['year'].isin([2014, 2015])]

# Get the list of day columns ('d_1069', 'd_1070', etc.) for those two years
target_days = scoped_calendar['d'].tolist()

# Keep only the days that actually exist in your sales training data
day_cols = [d for d in target_days if d in sales.columns]

# Scope the Sales down to 1 Store (CA_1) and years 2014,2015
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
scoped_sales = sales[sales['store_id'] == 'CA_1'][id_cols + day_cols]

# Get the valid Walmart week IDs for 2014-2015
valid_weeks = scoped_calendar['wm_yr_wk'].unique()

# Filter prices for CA_1 and those specific weeks
scoped_prices = prices[(prices['store_id'] == 'CA_1') & (prices['wm_yr_wk'].isin(valid_weeks))]

print(f"New Scoped Sales Shape: {scoped_sales.shape}")
print(f"New Scoped Calendar Shape: {scoped_calendar.shape}")
print(f"New Scoped Price Shape: {scoped_prices.shape}")

In [ ]:
# Let's see how much did we reduce.
# 1. Compare Sales Shapes
print("SALES TABLE REDUCTION")
print(f"Original Sales Shape: {sales.shape[0]:,} rows x {sales.shape[1]:,} columns")
print(f"Scoped Sales Shape:   {scoped_sales.shape[0]:,} rows x {scoped_sales.shape[1]:,} columns")
print(f"Row reduction:        {((1 - scoped_sales.shape[0]/sales.shape[0]) * 100):.1f}% fewer rows")
print(f"Column reduction:     {((1 - scoped_sales.shape[1]/sales.shape[1]) * 100):.1f}% fewer columns")

# Compare Calendar Shapes
print("\nCALENDAR TABLE REDUCTION")
print(f"Original Calendar Rows: {calendar.shape[0]:,}")
print(f"Scoped Calendar Rows:   {scoped_calendar.shape[0]:,}")
print(f"Row reduction:          {((1 - scoped_calendar.shape[0]/calendar.shape[0]) * 100):.1f}% fewer days")


# 2. Compare Prices Shapes
print("\nPRICES TABLE REDUCTION")
print(f"Original Prices Rows: {prices.shape[0]:,}")
print(f"Scoped Prices Rows:   {scoped_prices.shape[0]:,}")
print(f"Row reduction:        {((1 - scoped_prices.shape[0]/prices.shape[0]) * 100):.1f}% fewer price logs")


Before merging the tables, I will use `melt()` to reshape the sales table. Each resulting row will represent **one item at one store on one day**.

In [ ]:
# Reshape tiny subset for preview

tiny_wide = scoped_sales[id_cols + day_cols[:3]].head(2)
print("Before: 2 rows, 3 day columns")
display(tiny_wide)
tiny_long = tiny_wide.melt(id_vars=id_cols, var_name="d", value_name="sales")
print(f"After: {len(tiny_long)} rows, one row per item-store-day")
display(tiny_long)

In [ ]:
# Reshape whole subset

sales_long = scoped_sales.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales")
print(sales_long.shape)
display(sales_long.head())
display(sales_long["sales"].describe())


In [ ]:
# Merge with scoped_calendar table
sales_with_calendar = pd.merge(
    sales_long, 
    scoped_calendar, 
    on='d', 
    how='left',
    validate='many_to_one',
    indicator='cal_match'
)
display(sales_with_calendar['cal_match'].value_counts())

In [ ]:
sales_with_calendar = sales_with_calendar.drop(columns='cal_match')
print("Dropped cal match")

In [ ]:
# Merge with scoped_prices table
df = pd.merge(
    sales_with_calendar, 
    scoped_prices, 
    on=['store_id', 'item_id', 'wm_yr_wk'], 
    how='left',
    validate='many_to_one',
    indicator='price_match'
)
display(df['price_match'].value_counts())

In [ ]:
# Keep a compact missing-price flag for investigation in the next notebook.
df["price_missing"] = df["price_match"].eq("left_only")
df = df.drop(columns="price_match")
assert len(df) == len(sales_long)
assert not df.duplicated(["item_id", "store_id", "date"]).any()
assert df["date"].notna().all()
print("Joins preserved the daily rows and unique item-store-date keys.")

## Inspect the combined table

In [ ]:
print("Combined shape :", df.shape)
print("Date range     :", df["date"].min(), "to", df["date"].max())
display(df.head())

In [ ]:
df.info()

In [ ]:
df[['sales', 'sell_price']].describe()

In [ ]:
# Check missing values after merged.
missing_summary = pd.DataFrame({"count": df.isna().sum(),"percent": (df.isna().mean() * 100).round(2)})
display(missing_summary.loc[missing_summary["count"] > 0])

Now we see missing selling prices after the merge. Could these be periods before an item was offered, or could there be another explanation? I will inspect the sales values before deciding what to do.

In [ ]:
null_prices = df[df['sell_price'].isna()]

display(null_prices[['date', 'item_id', 'sales']].head(10))
display(null_prices[['date', 'item_id', 'sales']].tail())


In [ ]:
null_prices['sales'].value_counts()

All 75,616 rows without matching prices had **recorded sales of zero** in my original run. These are not missing sales values. The M5 guide says a missing price means the product was not sold during the examined week; this does not tell me exactly why.

I should not assume that staff created the item and added its price later. Zero sales could reflect no demand or limited availability, and these files do not establish daily stock availability or lost demand.

For now, I will **keep these rows** and mark the missing prices. Removing them would change zero-sales proportions and could leave gaps in the daily series. Before any exclusion or price imputation, I need to investigate the timing of the missing records and define a clear rule. I will not replace missing prices with zero.

In [ ]:
# Preserve every recorded sales day. Do not delete missing-price rows.
assert df["price_missing"].equals(df["sell_price"].isna())
print("Rows retained:", len(df))
print("Rows without a price:", df["price_missing"].sum())

In [ ]:
# Check rather than assume that unmatched prices have zero sales.
print("Sales values on missing-price days:")
display(df.loc[df["price_missing"], "sales"].value_counts(dropna=False))

In [ ]:
# Sort by series and actual date before future lag/rolling calculations.
df = df.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)
assert len(df) == len(scoped_sales) * len(day_cols)
print("Final shape:", df.shape)
print("Memory used by df (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 1))

## Clean up working variables

The joined `df` now contains the data I need for the next step. I will remove the original tables and temporary previews to release unused references. This does not delete any files. Python may retain some allocated memory, so the operating system may not show an immediate drop of the same size.

I will keep `df`, `DATA_DIR`, and the imported tools. To inspect the original tables again, I can rerun the loading cells.

In [ ]:
# Delete only known temporary variables; this cell is safe to run again.
for variable_name in [
    "sales", "calendar", "prices", "sales_calendar", "sales_day_cols",
    "scoped_sales", "scoped_calendar", "scoped_prices", "sales_long",
    "sales_with_calendar", "tiny_wide", "tiny_long", "null_prices",
    "missing_summary", "missing_counts", "actual_missing", "sales_cat_cols",
    "target_days", "day_cols", "id_cols", "valid_weeks", "table", "name", "col",
]:
    globals().pop(variable_name, None)
del variable_name
gc.collect()
print("Temporary tables cleared. df is ready to save.")

## Save the prepared data for the next notebook

I will save `df` as a **compressed CSV** in the data folder. It works with pandas without an extra storage library and is smaller than an uncompressed CSV. Missing prices remain blank, and `price_missing` records which rows lack a price.

Running this cell again replaces the same prepared file. The raw input CSV files are unchanged. Unlike some binary formats, CSV does not preserve every pandas dtype, so I will explicitly parse `date` when loading it again.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = DATA_DIR / "m5_ca1_2014_2015_prepared.csv.gz"
df.to_csv(output_path, index=False, compression="gzip")
print("Saved:", output_path.resolve())
print("Rows:", f"{len(df):,}")
print("File size (MB):", round(output_path.stat().st_size / 1024**2, 2))

## Start the next notebook

Upload or copy the saved `m5_ca1_2014_2015_prepared.csv.gz` file into the data folder used by the next notebook. Then load it with:

```python
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")  # Adjust to the folder containing the uploaded file.
df = pd.read_csv(
    DATA_DIR / "m5_ca1_2014_2015_prepared.csv.gz",
    parse_dates=["date"],
    dtype={"price_missing": "boolean"},
)
print(df.shape)
df.head()
```